In [1]:
import ee, json, random, math, csv, os
from kaggle_secrets import UserSecretsClient

key = UserSecretsClient().get_secret("EE_KEY")
email = json.loads(key)["client_email"]
ee.Initialize(ee.ServiceAccountCredentials(email, key_data=key), project="lulc-sar-project")

rng = random.Random(42)
TOTAL, MIN_DIST_KM, PER_RUN = 1000, 25, 125

HAND = [
 ("Heidelberg",49.40,8.69),("Milan",45.46,9.19),("LochNess",57.30,-4.45),("Almeria",36.80,-2.70),
 ("Kiruna",67.85,20.22),("Delhi",28.61,77.21),("Shanghai",31.23,121.47),("Borneo",0.50,114.00),
 ("Riyadh",24.71,46.68),("Yakutsk",62.03,129.73),("Everest",27.99,86.93),("MekongDelta",10.00,105.80),
 ("Cairo",30.04,31.24),("Niamey",13.51,2.11),("Kisangani",0.52,25.19),("Serengeti",-2.33,34.83),
 ("CapeTown",-33.92,18.42),("LacLaRonge",55.10,-105.30),("Iowa",42.00,-93.60),("Phoenix",33.45,-112.07),
 ("MexicoCity",19.43,-99.13),("Fairbanks",64.84,-147.72),("Everglades",25.75,-80.90),("Manaus",-3.12,-60.02),
 ("Titicaca",-15.80,-69.40),("BuenosAires",-34.60,-58.38),("SaoPaulo",-23.55,-46.63),("Uluru",-25.34,131.04),
 ("Sydney",-33.87,151.21),("MountCook",-43.59,170.14),("Tokyo",35.68,139.69),("Gobi",43.50,104.00),
 ("Jakarta",-6.20,106.85),("Jambi",-1.60,103.60),("CostaRica",10.40,-84.00),("Marseille",43.30,5.37),
 ("Athens",37.98,23.73),("Windhoek",-22.56,17.08),("Okavango",-19.30,22.90),("Tromso",69.65,18.96),
 ("Nairobi",-1.29,36.82),("Lagos",6.52,3.38),("Tehran",35.69,51.39),("Dubai",25.20,55.27),
 ("Jaisalmer",26.92,70.91),("Sundarbans",22.00,89.20),("Lhasa",29.65,91.10),("Beijing",39.90,116.40),
 ("Chicago",41.88,-87.63),("ColoradoRockies",39.60,-106.00),("Antofagasta",-23.65,-70.40),
 ("Pantanal",-17.80,-57.40),("Perth",-31.95,115.86),("Cairns",-16.92,145.77),("Vatnajokull",64.40,-16.80),
]
BOXES = [(36,70,-10,40,8),(8,70,40,145,25),(-9,20,92,141,6),(-34,35,-17,50,20),
         (15,70,-168,-55,18),(7,23,-92,-77,2),(-55,12,-80,-35,14),(-44,-10,113,154,7)]

def hav(a, b, c, d):
    p = math.pi/180
    x = math.sin((c-a)*p/2)**2 + math.cos(a*p)*math.cos(c*p)*math.sin((d-b)*p/2)**2
    return 12742*math.asin(math.sqrt(x))

wc = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
cands = []
for _ in range(9000):
    b = rng.choices(BOXES, [x[4] for x in BOXES])[0]
    cands.append((round(rng.uniform(b[0], b[1]), 3), round(rng.uniform(b[2], b[3]), 3)))

land = []
for s in range(0, len(cands), 3000):
    chunk = cands[s:s+3000]
    fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lo, la]), {"k": j}) for j, (la, lo) in enumerate(chunk)])
    res = wc.reduceRegions(fc, ee.Reducer.first(), 10).getInfo()["features"]
    vals = {f["properties"]["k"]: f["properties"].get("first") for f in res}
    land += [c for j, c in enumerate(chunk) if vals.get(j) not in (None, 80)]
print("land candidates:", len(land))

places = [(n, la, lo) for n, la, lo in HAND]
for la, lo in land:
    if len(places) >= TOTAL: break
    if all(hav(la, lo, p[1], p[2]) >= MIN_DIST_KM for p in places):
        places.append((f"R{len(places)+1:04d}", la, lo))
print("places:", len(places))

os.makedirs("/kaggle/working", exist_ok=True)
with open("/kaggle/working/places.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["id","name","lat","lon","split","run"])
    for i, (n, la, lo) in enumerate(places, 1):
        r = rng.random()
        split = "test" if r < 0.15 else ("val" if r < 0.20 else "train")
        w.writerow([i, n, la, lo, split, (i-1)//PER_RUN + 1])
print("saved places.csv")

land candidates: 4812
places: 1000
saved places.csv


In [2]:
import ee, json, os, math, io, time, requests
import numpy as np, pandas as pd, rasterio
from concurrent.futures import ThreadPoolExecutor
from kaggle_secrets import UserSecretsClient

key = UserSecretsClient().get_secret("EE_KEY")
email = json.loads(key)["client_email"]
ee.Initialize(ee.ServiceAccountCredentials(email, key_data=key), project="lulc-sar-project")

RUN = 1
START_Y, START_M, N_MONTHS = 2025, 1, 21
KM, P = 10, 256
OUT = f"/kaggle/working/run{RUN}"
os.makedirs(OUT, exist_ok=True)

places = pd.read_csv("/kaggle/working/places.csv")
places = places[places.run == RUN]
print("places in this run:", len(places))

def utm(lat, lon):
    return f"EPSG:{(32600 if lat >= 0 else 32700) + int((lon + 180)//6) + 1}"

def box(lat, lon):
    dlat = KM/2/111.32
    dlon = KM/2/(111.32*math.cos(math.radians(lat)))
    return ee.Geometry.Rectangle([lon-dlon, lat-dlat, lon+dlon, lat+dlat])

def fetch(img, region, crs, tries=4):
    for k in range(tries):
        try:
            url = img.getDownloadURL({"region": region, "scale": 10, "crs": crs, "format": "GEO_TIFF"})
            r = requests.get(url, timeout=300); r.raise_for_status()
            with rasterio.open(io.BytesIO(r.content)) as src:
                return src.read()
        except Exception:
            time.sleep(4*(k+1))
    return None

def patches(arr):
    _, h, w = arr.shape
    return [arr[:, i:i+P, j:j+P] for i in range(0, h-P+1, P) for j in range(0, w-P+1, P)]

def do_place(row):
    pid = int(row.id); folder = f"{OUT}/{pid:04d}"
    if os.path.exists(f"{folder}/DONE"):
        return pid, "skipped"
    os.makedirs(folder, exist_ok=True)
    region, crs = box(row.lat, row.lon), utm(row.lat, row.lon)
    dw = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(region).filterDate("2025-01-01", "2026-10-01")
    lab = fetch(dw.select("label").mode().toByte(), region, crs)
    if lab is None:
        return pid, "no label"
    lp = patches(lab)
    n_ok = n_patch = 0
    for i in range(N_MONTHS):
        m = START_M - 1 + i; y, mo = START_Y + m//12, m % 12 + 1
        s = ee.Date.fromYMD(y, mo, 1)
        col = (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(region)
               .filterDate(s, s.advance(1, "month"))
               .filter(ee.Filter.eq("instrumentMode", "IW"))
               .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
               .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))
        if col.size().getInfo() == 0:
            continue
        arr = fetch(col.select(["VV", "VH"]).median().float(), region, crs)
        if arr is None:
            continue
        sp = patches(arr)
        k = min(len(sp), len(lp))
        np.savez(f"{folder}/{y}-{mo:02d}.npz", x=np.stack(sp[:k]).astype("float32"), y=np.stack(lp[:k]).astype("uint8"))
        n_ok += 1; n_patch += k
    open(f"{folder}/DONE", "w").write(f"{n_ok} months, {n_patch} patches")
    return pid, f"{n_ok} months, {n_patch} patches"

# TEST first: only 3 places. After it works, set TEST = False
TEST = False
rows = list(places.itertuples())[:3] if TEST else list(places.itertuples())
t0 = time.time()
with ThreadPoolExecutor(max_workers=4) as ex:
    for pid, msg in ex.map(do_place, rows):
        print(pid, msg)
print("done in", round((time.time()-t0)/60, 1), "min")

places in this run: 125
1 21 months, 189 patches
2 21 months, 189 patches
3 21 months, 252 patches
4 21 months, 189 patches
5 21 months, 189 patches
6 21 months, 189 patches
7 20 months, 180 patches
8 21 months, 189 patches
9 21 months, 189 patches
10 14 months, 126 patches
11 21 months, 189 patches
12 21 months, 189 patches
13 21 months, 189 patches
14 21 months, 189 patches
15 21 months, 189 patches
16 21 months, 189 patches
17 21 months, 252 patches
18 18 months, 162 patches
19 21 months, 189 patches
20 21 months, 189 patches
21 21 months, 189 patches
22 21 months, 189 patches
23 21 months, 189 patches
24 21 months, 189 patches
25 21 months, 189 patches
26 21 months, 189 patches
27 21 months, 189 patches
28 19 months, 171 patches
29 21 months, 189 patches
30 21 months, 189 patches
31 21 months, 189 patches
32 20 months, 180 patches
33 21 months, 189 patches
34 21 months, 189 patches
35 21 months, 189 patches
36 21 months, 336 patches
37 21 months, 336 patches
38 21 months, 189 patch